In [1]:
!pip install python-chess


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import hashlib
from pathlib import Path

import chess.pgn
import pandas as pd
from tqdm import tqdm

DATA_DIR = Path("Data")

games = []
seen_games = set()

game_id = 1

pgn_files = sorted(DATA_DIR.glob("*.pgn"))

for pgn_file in tqdm(pgn_files, desc="PGN Files"):

    with open(pgn_file, encoding="utf-8", errors="ignore") as f:

        while True:
            game = chess.pgn.read_game(f)

            if game is None:
                break

            headers = game.headers

            # Full move sequence
            moves = [move.uci() for move in game.mainline_moves()]
            moves_str = " ".join(moves)

            row = {
                "game_id": game_id,
                "source_file": pgn_file.name,

                "event": headers.get("Event", ""),
                "site": headers.get("Site", ""),
                "date": headers.get("Date", ""),
                "round": headers.get("Round", ""),

                "white_player": headers.get("White", ""),
                "black_player": headers.get("Black", ""),

                "white_elo": headers.get("WhiteElo", ""),
                "black_elo": headers.get("BlackElo", ""),

                "eco": headers.get("ECO", ""),
                "result": headers.get("Result", ""),

                "moves": moves_str,
                "num_moves": len(moves),
            }

            # ===================================================
            # Exact duplicate detection
            # ===================================================

            fingerprint_fields = [
                row["event"],
                row["site"],
                row["date"],
                row["round"],
                row["white_player"],
                row["black_player"],
                row["white_elo"],
                row["black_elo"],
                row["eco"],
                row["result"],
                row["moves"]
            ]

            fingerprint = hashlib.sha256(
                "||".join(map(str, fingerprint_fields)).encode()
            ).hexdigest()

            if fingerprint in seen_games:
                continue

            seen_games.add(fingerprint)

            row["game_hash"] = fingerprint

            games.append(row)
            game_id += 1

games_df = pd.DataFrame(games)

print(f"Unique games: {len(games_df):,}")

games_df.head()

PGN Files: 100%|██████████| 7/7 [01:39<00:00, 14.17s/it]

Unique games: 33,129


,game_id,source_file,event,site,date,round,white_player,black_player,white_elo,black_elo,eco,result,moves,num_moves,game_hash
0,1,Anand.pgn,Wch U20,Kiljava,1984.??.??,?,"Anand, Viswanathan","Wolff, Patrick G",2285,2225,B09,0-1,e2e4 d7d6 d2d4 g8f6 b1c3 g7g6 f2f4 f8g7 g1f3 e...,44,d7afc4069a284a27c6adfa610e3c320c19bf4e43bc286a...
1,2,Anand.pgn,Lloyds Bank op,London,1984.??.??,5,"Chandler, Murray G","Anand, Viswanathan",2540,2345,B66,1-0,e2e4 c7c5 g1f3 b8c6 d2d4 c5d4 f3d4 g8f6 b1c3 d...,75,fae4385aeea9ff2fa6b6fd74591d347dec4b2e52645e43...
2,3,Anand.pgn,Lloyds Bank op,London,1984.??.??,8,"Anand, Viswanathan","Wells, Peter K",2345,2350,B17,1/2-1/2,e2e4 c7c6 d2d4 d7d5 b1c3 d5e4 c3e4 b8d7 g1f3 g...,46,57ad615d4794a46dd1182f0eec864bcb640ea839dc1720...
3,4,Anand.pgn,Lloyds Bank op,London,1984.??.??,9,"Anand, Viswanathan","Greenfeld, Alon",2345,2485,B49,1-0,e2e4 c7c5 g1f3 e7e6 b1c3 a7a6 d2d4 c5d4 f3d4 d...,159,d3c3f807486f9c2a27e6cbcdb9dbb7e766b05415408028...
4,5,Anand.pgn,Thessaloniki ol (Men),Thessaloniki,1984.??.??,1,"Barumba, G.","Anand, Viswanathan",,2345,E94,0-1,d2d4 g8f6 c2c4 g7g6 b1c3 f8g7 g1f3 e8g8 e2e4 d...,82,f81a6dedd8b030dbb7cb4790b3eb1ca9bcab17fcedb91d...


In [3]:
games_df_backup=games_df.copy()
games_df = games_df[
    ["game_id", "moves", "result"]
].copy()

In [4]:
# print(len(games_df))
# print(games_df["num_moves"].describe())


In [5]:
import chess
import pandas as pd
from tqdm import tqdm

def generate_positions(games_df):

    rows = []

    for row in tqdm(
        games_df.itertuples(index=False),
        total=len(games_df)
    ):
        board = chess.Board()

        moves = row.moves.split()

        for ply, move in enumerate(moves):

            rows.append(
                (
                    row.game_id,
                    ply,
                    board.board_fen(),   # smaller than full fen
                    move,
                    row.result
                )
            )

            try:
                board.push_uci(move)

            except:
                break

    return pd.DataFrame(
        rows,
        columns=[
            "game_id",
            "ply",
            "board_fen",
            "move",
            "result"
        ]
    )

In [6]:
positions_df = generate_positions(games_df)

print(positions_df.shape)
positions_df.head()

100%|██████████| 33129/33129 [05:36<00:00, 98.40it/s] 


(2967604, 5)


,game_id,ply,board_fen,move,result
0,1,0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR,e2e4,0-1
1,1,1,rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR,d7d6,0-1
2,1,2,rnbqkbnr/ppp1pppp/3p4/8/4P3/8/PPPP1PPP/RNBQKBNR,d2d4,0-1
3,1,3,rnbqkbnr/ppp1pppp/3p4/8/3PP3/8/PPP2PPP/RNBQKBNR,g8f6,0-1
4,1,4,rnbqkb1r/ppp1pppp/3p1n2/8/3PP3/8/PPP2PPP/RNBQKBNR,b1c3,0-1


In [7]:
print("Total positions:", len(positions_df))
print("Unique positions:", positions_df["board_fen"].nunique())

Total positions: 2967604
Unique positions: 2414427


In [8]:
position_counts = (
    positions_df["board_fen"]
    .value_counts()
)

print(position_counts.head(20))

board_fen
rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR             33123
rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR           14139
rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR           10582
rnbqkb1r/pppppppp/5n2/8/3P4/8/PPP1PPPP/RNBQKBNR          6947
rnbqkbnr/pp1ppppp/8/2p5/4P3/8/PPPP1PPP/RNBQKBNR          5339
rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR          5230
rnbqkb1r/pppppppp/5n2/8/2PP4/8/PP2PPPP/RNBQKBNR          5063
rnbqkbnr/pppp1ppp/8/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R        4806
rnbqkbnr/pppppppp/8/8/8/5N2/PPPPPPPP/RNBQKB1R            4760
rnbqkbnr/pp1ppppp/8/2p5/4P3/5N2/PPPP1PPP/RNBQKB1R        4533
r1bqkbnr/pppp1ppp/2n5/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R      4329
rnbqkb1r/pppp1ppp/4pn2/8/2PP4/8/PP2PPPP/RNBQKBNR         3584
r1bqkbnr/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQK2R     2759
rnbqkb1r/pppp1ppp/4pn2/8/2PP4/5N2/PP2PPPP/RNBQKB1R       2478
rnbqkbnr/pppppppp/8/8/2P5/8/PP1PPPPP/RNBQKBNR            2378
rnbqkbnr/ppp1pppp/8/3p4/3P4/8/PPP1PPPP/RNBQKBNR          235

In [9]:
import chess

def fen_to_tokens(board_fen):

    board = chess.Board()

    board.set_board_fen(board_fen)

    tokens = []

    for square, piece in board.piece_map().items():

        color = "W" if piece.color else "B"

        piece_symbol = piece.symbol().upper()

        square_name = chess.square_name(square)

        tokens.append(
            f"{color}{piece_symbol}_{square_name}"
        )

    return tokens

In [10]:
tokens = (
    positions_df["board_fen"]
    
    .apply(fen_to_tokens)
)

tokens.head()

0    [BR_h8, BN_g8, BB_f8, BK_e8, BQ_d8, BB_c8, BN_...
1    [BR_h8, BN_g8, BB_f8, BK_e8, BQ_d8, BB_c8, BN_...
2    [BR_h8, BN_g8, BB_f8, BK_e8, BQ_d8, BB_c8, BN_...
3    [BR_h8, BN_g8, BB_f8, BK_e8, BQ_d8, BB_c8, BN_...
4    [BR_h8, BB_f8, BK_e8, BQ_d8, BB_c8, BN_b8, BR_...
Name: board_fen, dtype: object

In [11]:
fen_to_tokens(
    positions_df.iloc[0]["board_fen"]
)

['BR_h8',
 'BN_g8',
 'BB_f8',
 'BK_e8',
 'BQ_d8',
 'BB_c8',
 'BN_b8',
 'BR_a8',
 'BP_h7',
 'BP_g7',
 'BP_f7',
 'BP_e7',
 'BP_d7',
 'BP_c7',
 'BP_b7',
 'BP_a7',
 'WP_h2',
 'WP_g2',
 'WP_f2',
 'WP_e2',
 'WP_d2',
 'WP_c2',
 'WP_b2',
 'WP_a2',
 'WR_h1',
 'WN_g1',
 'WB_f1',
 'WK_e1',
 'WQ_d1',
 'WB_c1',
 'WN_b1',
 'WR_a1']

In [12]:
from collections import Counter

def build_vocab(token_lists):

    vocab = {}

    idx = 0

    for tokens in token_lists:

        for token in tokens:

            if token not in vocab:

                vocab[token] = idx
                idx += 1

    return vocab

In [15]:
# 1. Generate the token lists from the FEN strings
tokens = positions_df["board_fen"].apply(fen_to_tokens)

# 2. BUILD THE VOCAB HERE
vocab = build_vocab(tokens)
print("Vocab Size:", len(vocab))
print(list(vocab.items())[:20])

# 3. Map tokens to IDs using the newly created vocab
def tokens_to_ids(tokens):
    return [vocab[token] for token in tokens]

positions_df["token_ids"] = tokens.apply(tokens_to_ids) # Reusing the 'tokens' variable is faster than calling fen_to_tokens again!

Vocab Size: 736
[('BR_h8', 0), ('BN_g8', 1), ('BB_f8', 2), ('BK_e8', 3), ('BQ_d8', 4), ('BB_c8', 5), ('BN_b8', 6), ('BR_a8', 7), ('BP_h7', 8), ('BP_g7', 9), ('BP_f7', 10), ('BP_e7', 11), ('BP_d7', 12), ('BP_c7', 13), ('BP_b7', 14), ('BP_a7', 15), ('WP_h2', 16), ('WP_g2', 17), ('WP_f2', 18), ('WP_e2', 19)]


In [16]:
print(len(vocab))

736


In [17]:
list(vocab.items())[:20]

[('BR_h8', 0),
 ('BN_g8', 1),
 ('BB_f8', 2),
 ('BK_e8', 3),
 ('BQ_d8', 4),
 ('BB_c8', 5),
 ('BN_b8', 6),
 ('BR_a8', 7),
 ('BP_h7', 8),
 ('BP_g7', 9),
 ('BP_f7', 10),
 ('BP_e7', 11),
 ('BP_d7', 12),
 ('BP_c7', 13),
 ('BP_b7', 14),
 ('BP_a7', 15),
 ('WP_h2', 16),
 ('WP_g2', 17),
 ('WP_f2', 18),
 ('WP_e2', 19)]

In [18]:
import torch
import torch.nn as nn

In [19]:
embedding = nn.Embedding(
    num_embeddings=736,
    embedding_dim=128
)

SAVING

In [20]:
games_df.to_parquet("Data\games.parquet", index=False)
positions_df.to_parquet("Data\positions.parquet", index=False)

<>:1: SyntaxWarning: invalid escape sequence '\g'
<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\g'
<>:2: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Admin\AppData\Local\Temp\ipykernel_30408\3954419252.py:1: SyntaxWarning: invalid escape sequence '\g'
  games_df.to_parquet("Data\games.parquet", index=False)
C:\Users\Admin\AppData\Local\Temp\ipykernel_30408\3954419252.py:2: SyntaxWarning: invalid escape sequence '\p'
  positions_df.to_parquet("Data\positions.parquet", index=False)


REUSE Directly below this df's by importing saves

In [21]:
def tokens_to_ids(tokens):
    return [vocab[token] for token in tokens]

positions_df["token_ids"] = (
    positions_df["board_fen"]
    .apply(fen_to_tokens)
    .apply(tokens_to_ids)
)

In [ ]:
# # After your expensive computation
# positions_df.to_parquet("Data/positions_processed.parquet", index=False)

# # Later, just load it back
# positions_df = pd.read_parquet("Data/positions_processed.parquet")

In [ ]:
# positions_df = pd.read_parquet("Data/positions_processed.parquet")

In [22]:
positions_df["token_ids"].head()

0    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
1    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
2    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14,...
3    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14,...
4    [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15...
Name: token_ids, dtype: object

In [23]:
all_moves = sorted(
    positions_df["move"].unique()
)

move_vocab = {
    move: idx
    for idx, move in enumerate(all_moves)
}

idx_to_move = {
    idx: move
    for move, idx in move_vocab.items()
}

In [43]:
print(len(positions_df))

2967604


In [ ]:
positions_df["move_id"] = (
    positions_df["move"]
    .map(move_vocab)
)

Unique moves: 1896


In [26]:
# PAD TOKEN SEQUENCES TO 32 since NNs want fixed
MAX_PIECES = 32

PAD_ID = len(vocab)

vocab_size = PAD_ID + 1

In [ ]:
import numpy as np

def pad_token_sequences(token_series):

    token_lists = token_series.tolist()

    X = np.full(
        (len(token_lists), MAX_PIECES),
        PAD_ID,
        dtype=np.int64
    )

    for i, tokens in enumerate(token_lists):

        n = min(len(tokens), MAX_PIECES)

        X[i, :n] = tokens[:n]

    return X


# Create padded training matrix
X = pad_token_sequences(
    positions_df["token_ids"]
)

print("X shape:", X.shape)
print(X[0])


X shape: (2967604, 32)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31]


In [48]:

print("X shape:", X.shape)
print(X[20000])

X shape: (2967604, 32)
[ 41  42   5   7   8   9  14  15  35  48 452  53 540  91  58  40  54  56
 131  16  17  18  22  23 123 132  28  31 736 736 736 736]


In [39]:
print(
    positions_df["token_ids"].iloc[0].shape
)

print(
    positions_df["token_ids"].iloc[0]
)

(32,)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31]


In [46]:
positions_df["token_ids"].apply(len).describe()

count    2.967604e+06
mean     2.226648e+01
std      7.863902e+00
min      2.000000e+00
25%      1.600000e+01
50%      2.400000e+01
75%      3.000000e+01
max      3.200000e+01
Name: token_ids, dtype: float64

STEP 4 CREATE Tensors

In [51]:
import numpy as np
import torch


y = positions_df["move_id"].values

X = torch.tensor(
    X,
    dtype=torch.long
)

y = torch.tensor(
    y,
    dtype=torch.long
)

In [52]:
print(X.shape)
print(y.shape)

torch.Size([2967604, 32])
torch.Size([2967604])


In [53]:
print(X.nbytes / 1024**3)
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

0.7075319290161133
False


In [54]:
unique_games = positions_df["game_id"].unique()

In [55]:
print(unique_games)

[    1     2     3 ... 33127 33128 33129]


# BUILD V2 MODEL



In [109]:
import os
import torch

NUM_WORKERS = max(
    1,
    os.cpu_count()-8
)
NUM_WORKERS=0
torch.set_num_threads(
    os.cpu_count()
)

print("CPU cores:", os.cpu_count())
print("Workers:", NUM_WORKERS)

CPU cores: 12
Workers: 0


In [110]:
from sklearn.model_selection import train_test_split

unique_games = (
    positions_df["game_id"]
    .unique()
)

train_games, val_games = train_test_split(
    unique_games,
    test_size=0.1,
    random_state=42
)

train_mask = (
    positions_df["game_id"]
    .isin(train_games)
)

val_mask = (
    positions_df["game_id"]
    .isin(val_games)
)

In [111]:
X_train = X[train_mask.values]
y_train = y[train_mask.values]

X_val = X[val_mask.values]
y_val = y[val_mask.values]

print(X_train.shape)
print(X_val.shape)

torch.Size([2669196, 32])
torch.Size([298408, 32])


In [112]:
from torch.utils.data import Dataset
#4 Dataset

class ChessDataset(Dataset):

    def __init__(self, X, y):

        self.X = X
        self.y = y

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.y[idx]
        )

In [113]:
train_dataset = ChessDataset(
    X_train,
    y_train
)

val_dataset = ChessDataset(
    X_val,
    y_val
)

print(len(train_dataset))
print(len(val_dataset))

2669196
298408


In [114]:
from torch.utils.data import DataLoader

BATCH_SIZE = 2048

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

In [115]:
# 6 ACTUAL V2 Model

In [116]:
import torch.nn as nn

class ChessEmbeddingModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        num_moves,
        embedding_dim=128
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )

        self.fc = nn.Sequential(

            nn.Linear(
                embedding_dim,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                256,
                num_moves
            )
        )

    def forward(self, x):

        x = self.embedding(x)

        mask = (
            x.abs()
            .sum(dim=-1)
            != 0
        )

        x = (
            x * mask.unsqueeze(-1)
        )

        counts = (
            mask.sum(dim=1)
            .clamp(min=1)
            .unsqueeze(-1)
        )

        x = (
            x.sum(dim=1)
            / counts
        )

        return self.fc(x)

In [117]:
device = torch.device("cpu")

model = ChessEmbeddingModel(
    vocab_size=vocab_size,
    num_moves=len(move_vocab)
)

model.to(device)

ChessEmbeddingModel(
  (embedding): Embedding(737, 128, padding_idx=736)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=1896, bias=True)
  )
)

In [118]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

In [119]:
batch_x, batch_y = next(iter(train_loader))

print(batch_x.shape)
print(batch_y.shape)

torch.Size([2048, 32])
torch.Size([2048])


In [120]:
from tqdm import tqdm

model.train()

for batch_x, batch_y in tqdm(train_loader):

    batch_x = batch_x.to(device)

    batch_y = batch_y.to(device)

    optimizer.zero_grad()

    logits = model(batch_x)

    loss = criterion(
        logits,
        batch_y
    )

    loss.backward()

    optimizer.step()

print(
    "Final loss:",
    loss.item()
)

100%|██████████| 1304/1304 [04:29<00:00,  4.83it/s]

Final loss: 5.0255351066589355


In [121]:
embeddings = (
    model.embedding
    .weight
    .detach()
    .cpu()
)

torch.save(
    embeddings,
    "piece_embeddings.pt"
)

print(
    embeddings.shape
)

torch.Size([737, 128])


In [123]:
import torch

# Load the embedding file
data = torch.load('piece_embedding.pt')

# Check what type of data is inside (e.g., dict, tensor)
print(type(data))

# If it's a tensor, check its shape
if isinstance(data, torch.Tensor):
    print("Tensor shape:", data.shape)
elif isinstance(data, dict):
    print("Keys in the dictionary:", data.keys())

FileNotFoundError: [Errno 2] No such file or directory: 'piece_embedding.pt'